# Fitting the GRB Spectrum with the Binned Relative-Coordinates Response

## Introduction

This notebook is a companion to
[example_grb_fit_normalizing_flows.ipynb](example_grb_fit_normalizing_flows.ipynb), which
introduces the classes used for unbinned spectral fitting (`UnbinnedThreeMLPointSourceResponseIRFAdaptive`,
`CachedUnbinnedThreeMLModelFolding`, etc.) -- see that notebook for details on those. Here we
only cover what's different: the response itself.

`IRFRelativeHistUnpolarized` evaluates the response by interpolating a 6D histogram
(`NuLambda, Ei, Epsilon, Phi, Theta, Zeta`) binned in *relative* coordinates (relative to the
photon's incoming direction and energy), instead of evaluating a neural network. It doesn't need
a GPU and is much cheaper to query, at the cost of some accuracy (see
[this presentation](https://github.com/user-attachments/files/31315263/20260428-BinnedRelResponse-cosipy-Israel.pdf)
for how it's built and a first comparison against the neural-network response).

This notebook demonstrates two ways of building that histogram, selected below via `irf_mode`:

- `"hist_simple"`: the histogram built directly from a MEGAlib simulation binned in relative
  coordinates (see `cosipy/response/scripts/IRFRelativeHist/relative_hist_irf_from_rsp.py`).
  This is the response you'd use for an actual analysis.
- `"hist_nn"`: the same kind of histogram, but built by evaluating the neural-network response
  on a grid instead of from a MEGAlib simulation (see
  `cosipy/response/scripts/IRFRelativeHist/relative_hist_irf_from_nf_response.py`). Since it
  approximates the NN response rather than an independent simulation, it's mainly useful for
  validating the histogram-interpolation method itself against the NN response it was built
  from, decoupled from any difference between the NN and MEGAlib. Because it isn't tied to a
  MEGAlib simulation, it doesn't need the strip-discretization distance cut described below.

For reference, `"nn"` is also available, and simply reuses the neural-network response directly
(no histogram), exactly as in the other notebook.

### Choose the response mode

In [ ]:
irf_mode = "hist_simple"  # one of "hist_simple", "hist_nn", "nn"


### Basic Setup

In [ ]:
from pathlib import Path
from cosipy.util import fetch_wasabi_file
from cosipy.spacecraftfile import SpacecraftHistory
from astropy.time import Time

import astropy.units as u
from copy import deepcopy
import matplotlib.pyplot as plt
import numpy as np

from threeML import Band, PointSource, Model, JointLikelihood, DataList

from cosipy.threeml.unbinned_model_folding import CachedUnbinnedThreeMLModelFolding
from cosipy.statistics import UnbinnedLikelihood
from cosipy.interfaces import ThreeMLPluginInterface
from cosipy.interfaces.expectation_interface import SumExpectationDensity

from cosipy.event_selection import DistanceSelector, ChainEventSelectors
from cosipy.event_selection.time_selection import TimeSelector
from cosipy.data_io.EmCDSUnbinnedData import TimeTagEmCDSDistanceEventDataInSCFrameFromDC3Fits

from cosipy.response.ml.NFResponse import NFResponse
from cosipy.response.ml.nf_instrument_response_function import UnpolarizedNFFarFieldInstrumentResponseFunction
from cosipy.response.relative_irf_hist import IRFRelativeHistUnpolarized

from cosipy.threeml.ml.optimized_unbinned_folding import UnbinnedThreeMLPointSourceResponseIRFAdaptive


The GRB event data and spacecraft orientation are needed regardless of `irf_mode`; the
response file is not, so it's only fetched below for the mode actually selected above.

In [ ]:
data_path = Path("./") # Current path by default

grb_data_path = data_path / "GRB_bn090424592_3months_unbinned_data_filtered_with_SAAcut.fits.gz"
fetch_wasabi_file('COSI-SMEX/DC3/Data/Sources/GRB_bn090424592_3months_unbinned_data_filtered_with_SAAcut.fits.gz',
                  checksum = '08c1f1202d78df40a4aed3b785d273e0', output=str(grb_data_path))

sc_orientation_path = data_path / "DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits"
fetch_wasabi_file('COSI-SMEX/DC4/Data/Orientation/DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits',
                  checksum = 'ca94ff1d7a73c1f41479aaf598807673', output=str(sc_orientation_path))


In [ ]:
if irf_mode == "nn":

    rsp_path = data_path / "unpolarized_nfresponse_v1-01.pt"
    fetch_wasabi_file('COSI-SMEX/DC4/Data/Responses/unpolarized_nfresponse_v1-01.pt',
                      checksum = 'bf2d0c16eac5954fb56489480c2602ca', output=str(rsp_path))

elif irf_mode == "hist_simple":

    hist_zip_path = data_path / "ResponseContinuum.area.relative.nonsparse.h5.zip"
    hist_path = data_path / "ResponseContinuum.area.relative.nonsparse.h5"
    fetch_wasabi_file('COSI-SMEX/develop/Data/Responses/ResponseContinuum.area.relative.nonsparse.h5.zip',
                      output=str(hist_zip_path), unzip=True,
                      checksum = '2a5ab4f227608349f934d219ffebc2f4')

elif irf_mode == "hist_nn":

    hist_nn_zip_path = data_path / "relative_hist_irf_from_nf_response.h5.zip"
    hist_nn_path = data_path / "relative_hist_irf_from_nf_response.h5"
    fetch_wasabi_file('COSI-SMEX/develop/Data/Responses/relative_hist_irf_from_nf_response.h5.zip',
                      output=str(hist_nn_zip_path), unzip=True,
                      checksum = 'd9093daeaf56a386095a42ae40c6635e')

else:

    raise RuntimeError(f"irf_mode {irf_mode} is not supported.")


Define the observation duration here. This GRB lasts approximately 10 seconds.

In [ ]:
tstart = Time("2028-03-24 10:36:42.000")
tstop = Time("2028-03-24 10:36:51.921")
sc_orientation = SpacecraftHistory.open(sc_orientation_path)
sc_orientation = sc_orientation.select_interval(tstart, tstop)


Typically, you would need to perform a time selection. Although this tutorial includes
pre-selected files covering the full duration of the GRB, reducing the observation window is
recommended if you are working with limited computational resources.

`"hist_simple"` also needs a distance cut (first-two-hits distance > 1 cm): the MEGAlib
simulation it was built from used that same cut to avoid strip-discretization artifacts near the
detector edges, so the event data needs to match. `"hist_nn"` and `"nn"` don't need it, since
they aren't built from that simulation.

In [ ]:
time_selector = TimeSelector(tstart = sc_orientation.tstart, tstop = sc_orientation.tstop)

if irf_mode == "hist_simple":
    selector = ChainEventSelectors(DistanceSelector(min_distance=1 * u.cm), time_selector)
else:
    selector = time_selector

data = TimeTagEmCDSDistanceEventDataInSCFrameFromDC3Fits([grb_data_path], selection=selector)


In [ ]:
print(f"This analysis uses {data.nevents} Events")


### Building the response

- `"nn"` builds `NFResponse` as in the other notebook (see there for a description of its
  `devices`/`compile_mode` arguments) and wraps it in `UnpolarizedNFFarFieldInstrumentResponseFunction`.
- `"hist_simple"` and `"hist_nn"` just load the pre-built histogram with
  `IRFRelativeHistUnpolarized.from_h5()` -- no compute pool or GPU involved.

In [ ]:
if irf_mode == "nn":

    rsp = NFResponse(
        path_to_model=rsp_path,
        area_batch_size=300_000,
        density_batch_size=100_000,
        devices=["cpu"],
        area_compile_mode=None,
        density_compile_mode=None,
        show_progress=True)

    irf = UnpolarizedNFFarFieldInstrumentResponseFunction(rsp)

elif irf_mode == "hist_simple":

    irf = IRFRelativeHistUnpolarized.from_h5(hist_path)

elif irf_mode == "hist_nn":

    irf = IRFRelativeHistUnpolarized.from_h5(hist_nn_path)


In [ ]:
psr = UnbinnedThreeMLPointSourceResponseIRFAdaptive(
    data=data,
    irf=irf,
    sc_history=sc_orientation,
    show_progress=True,
    force_energy_node_caching=True,
    reduce_memory=True)

psr.cache_batch_size = 100_000
psr.integration_batch_size = 100_000


The default `Band`, `Powerlaw` and `PointSource` models can be slow during optimization (see
the [discussion here](https://github.com/cositools/cosipy/discussions/492)). For larger
datasets, consider implementing Torch-based versions to improve performance.

In [ ]:
l = 21.293418951586656
b = -42.384846594045484

alpha = -1.021866
beta = -2.762827
xp = 159.9995 * u.keV
piv = 100. * u.keV
K = 5.2385389556 / u.cm / u.cm / u.s / u.keV

spectrum = Band(beta=beta, K=K.value, piv=piv.value)

spectrum.alpha.delta = 0.01
spectrum.beta.delta = 0.01

spectrum.alpha.value = alpha
spectrum.xp.value = xp.value

spectrum.xp.unit = xp.unit
spectrum.K.unit = K.unit
spectrum.piv.unit = piv.unit


In [ ]:
spectrum_inj = deepcopy(spectrum)


In [ ]:
source = PointSource("GRB",
                     l=l,
                     b=b,
                     spectral_shape=spectrum)
model = Model(source)


In [ ]:
response = CachedUnbinnedThreeMLModelFolding(psr)


In [ ]:
expectation_density = SumExpectationDensity(response)


In [ ]:
like_fun = UnbinnedLikelihood(expectation_density)
cosi = ThreeMLPluginInterface('cosi', like_fun, response)


In [ ]:
plugins = DataList(cosi)
like = JointLikelihood(model, plugins, verbose=True) # You can disable debugging


### Initializing the Cache

Here you could load the cache

In [ ]:
# response.load_caches(data_path / "GRB_tutorial")


The cache is initialized, which takes some time

In [ ]:
print(f"Data Events: {data.nevents}\nExpected Events: {expectation_density.expected_counts():.2f}\nRelative Deviation {100 * (expectation_density.expected_counts()/data.nevents - 1):.3f} %")


Now you could save the cache.

In [ ]:
response.save_caches(data_path / "GRB_tutorial")


### Fitting

In [ ]:
like.fit()


Now we can plot the result and compare it with the injected spectrum.

In [ ]:
results = like.results

parameters = {par.name: results.get_variates(par.path)
              for par in results.optimized_model["GRB"].parameters.values()
              if par.free}

results_err = results.propagate(results.optimized_model["GRB"].spectrum.main.shape.evaluate_at, **parameters)


In [ ]:
energy = np.geomspace(100*u.keV, 10*u.MeV).to_value(u.keV)

flux_lo = np.zeros_like(energy)
flux_median = np.zeros_like(energy)
flux_hi = np.zeros_like(energy)
flux_inj = np.zeros_like(energy)

for i, e in enumerate(energy):
    flux = results_err(e)
    flux_median[i] = flux.median
    flux_lo[i], flux_hi[i] = flux.equal_tail_interval(cl=0.68)
    flux_inj[i] = spectrum_inj.evaluate_at(e)


In [ ]:
%matplotlib inline


In [ ]:
fig, ax = plt.subplots(figsize = (9, 6))

ax.plot(energy, energy**2 * flux_median, label = "Best fit")
ax.fill_between(energy, energy**2 * flux_lo, energy*energy*flux_hi, alpha = .5, label = "Best fit (errors)")
ax.plot(energy, energy**2 * flux_inj, color = 'black', ls = ":", label = "Injected")

ax.semilogx()
ax.semilogy()

ax.set_xlabel("Energy [keV]")
ax.set_ylabel(r"$E^2 \frac{\mathrm{d}N}{\mathrm{d}E}$ [keV cm$^{-2}$ s$^{-1}$]")

ax.legend();
